In [2]:
%pip install streamlit pandas scikit-learn

  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   -----------------------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd

df = pd.read_csv("tmdb_ds.csv")

print("Column names:")
print(df.columns.tolist())

Column names:
['movie_id', 'title', 'cast', 'crew']


In [5]:
df.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [6]:
import streamlit as st
import pandas as pd
import re
import ast
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(
    page_title="Movie Recommendation System",
    page_icon="🎬"
)

st.title("🎬 Movie Recommendation System")
st.write("Find movies with similar casts and crew!")

# Load dataset
@st.cache_data
def load_data():
    df = pd.read_csv("tmdb_ds.csv")

    df["title"] = df["title"].fillna("")
    df["cast"] = df["cast"].fillna("[]")
    df["crew"] = df["crew"].fillna("[]")

    return df

df = load_data()

# Extract names from the cast and crew JSON-like columns
def extract_names(value):
    try:
        people = ast.literal_eval(value)
        if isinstance(people, list):
            return " ".join(
                str(person.get("name", ""))
                for person in people
                if isinstance(person, dict)
            )
    except (ValueError, SyntaxError, TypeError):
        pass

    return ""

df["cast_names"] = df["cast"].apply(extract_names)
df["crew_names"] = df["crew"].apply(extract_names)

# Combine information into one text column
df["movie_features"] = (
    df["title"] + " " +
    df["cast_names"] + " " +
    df["crew_names"]
)

# Clean text
stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["clean_text"] = df["movie_features"].apply(clean_text)

# TF-IDF vectorization
@st.cache_data
def create_tfidf(text_data):
    vectorizer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2)
    )
    matrix = vectorizer.fit_transform(text_data)
    return matrix

tfidf_matrix = create_tfidf(df["clean_text"].tolist())

# Recommendation function
def recommend(item_name, top_n=5):
    matches = df.index[
        df["title"].str.strip().str.lower()
        == item_name.strip().lower()
    ].tolist()

    if not matches:
        return pd.DataFrame(
            columns=["title", "similarity_score"]
        )

    movie_index = matches[0]

    scores = cosine_similarity(
        tfidf_matrix[movie_index], tfidf_matrix
    ).flatten()

    similar_indices = scores.argsort()[::-1]
    similar_indices = [
        i for i in similar_indices if i != movie_index
    ][:top_n]

    return pd.DataFrame({
        "title": df.iloc[similar_indices]["title"].values,
        "similarity_score": scores[similar_indices]
    }).reset_index(drop=True)

# Movie selection
movie_list = sorted(df["title"].dropna().unique().tolist())

selected_movie = st.selectbox(
    "Select a movie:",
    movie_list
)

if st.button("Get Recommendations"):
    results = recommend(selected_movie, top_n=5)

    if results.empty:
        st.warning("No recommendations found.")
    else:
        st.subheader(f"Movies similar to {selected_movie}")

        for _, row in results.iterrows():
            st.write(f"🎥 **{row['title']}**")
            st.caption(
                f"Similarity score: {row['similarity_score']:.3f}"
            )


2026-10-19 21:03:00.040 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 21:03:00.041 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 21:03:00.041 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 21:03:00.041 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 21:03:00.043 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 21:03:00.043 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 21:03:00.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 21:03:00.046 No runtime found, using MemoryCacheStorageManager
2026-10-19 21:03:00.048 No runtime found, us